# Разведочный анализ данных (EDA)
## Проект: ProdRadar — мониторинг упоминаний бренда и анализ тональности

---

**Датасет:** `data/augmented_merged.jsonl` (~10K записей)  
**Поля:** `text` (текст отзыва), `sentiment` (метка тональности: позитив / негатив / нейтрально)  
**Задача:** Классификация тональности русскоязычных текстов (3 класса)

---

## 0. Импорт библиотек и настройка окружения

In [ ]:
import json
import re
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# --- настройка визуализации ---
sns.set_theme(style="whitegrid", font_scale=1.15)
rcParams["figure.dpi"] = 120
rcParams["axes.titlesize"] = 14
rcParams["axes.labelsize"] = 12

# палитра для трёх классов (негатив — красный, нейтрально — серый, позитив — зелёный)
SENTIMENT_COLORS = {
    "негатив": "#e74c3c",
    "нейтрально": "#95a5a6",
    "позитив": "#27ae60",
}
SENTIMENT_ORDER = ["негатив", "нейтрально", "позитив"]

print("Библиотеки загружены.")

---
## 1. Загрузка данных

In [ ]:
DATA_PATH = "data/augmented_merged.jsonl"

records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Загружено записей: {len(df):,}")
print(f"Столбцы: {list(df.columns)}")
print(f"Типы данных:\n{df.dtypes}")

In [ ]:
df.head(10)

In [ ]:
# Примеры из каждого класса
for label in SENTIMENT_ORDER:
    print(f"\n{'='*80}")
    print(f"Класс: {label.upper()}")
    print(f"{'='*80}")
    samples = df[df["sentiment"] == label]["text"].sample(3, random_state=42)
    for i, txt in enumerate(samples, 1):
        print(f"  {i}. {txt[:200]}{'...' if len(txt) > 200 else ''}")

---
## 2. Распределение классов

In [ ]:
class_counts = df["sentiment"].value_counts().reindex(SENTIMENT_ORDER)
class_pct = (class_counts / len(df) * 100).round(1)

print("Абсолютные значения:")
for label in SENTIMENT_ORDER:
    print(f"  {label:>12s}: {class_counts[label]:>5,}  ({class_pct[label]:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- столбчатая диаграмма ---
bars = axes[0].bar(
    SENTIMENT_ORDER,
    [class_counts[s] for s in SENTIMENT_ORDER],
    color=[SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER],
    edgecolor="white",
    linewidth=1.2,
)
for bar, label in zip(bars, SENTIMENT_ORDER):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{class_counts[label]:,}\n({class_pct[label]}%)",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )
axes[0].set_title("Распределение классов тональности")
axes[0].set_ylabel("Количество записей")
axes[0].set_ylim(0, class_counts.max() * 1.2)

# --- круговая диаграмма ---
wedges, texts, autotexts = axes[1].pie(
    [class_counts[s] for s in SENTIMENT_ORDER],
    labels=SENTIMENT_ORDER,
    colors=[SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER],
    autopct="%1.1f%%",
    startangle=140,
    textprops={"fontsize": 11},
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
for at in autotexts:
    at.set_fontweight("bold")
axes[1].set_title("Доля каждого класса")

plt.tight_layout()
plt.show()

---
## 3. Анализ длины текстов

In [ ]:
df["text_len"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().apply(len)

print("Общая статистика по длине текста (символы):")
df["text_len"].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- гистограмма длин в символах ---
for label in SENTIMENT_ORDER:
    subset = df[df["sentiment"] == label]["text_len"]
    axes[0].hist(
        subset, bins=50, alpha=0.55, label=label,
        color=SENTIMENT_COLORS[label], edgecolor="white", linewidth=0.5,
    )
axes[0].set_title("Распределение длины текста (символы)")
axes[0].set_xlabel("Длина текста (символы)")
axes[0].set_ylabel("Количество")
axes[0].legend()

# --- гистограмма количества слов ---
for label in SENTIMENT_ORDER:
    subset = df[df["sentiment"] == label]["word_count"]
    axes[1].hist(
        subset, bins=50, alpha=0.55, label=label,
        color=SENTIMENT_COLORS[label], edgecolor="white", linewidth=0.5,
    )
axes[1].set_title("Распределение количества слов")
axes[1].set_xlabel("Количество слов")
axes[1].set_ylabel("Количество")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(
    data=df, x="sentiment", y="text_len", order=SENTIMENT_ORDER,
    palette=SENTIMENT_COLORS, showfliers=False, width=0.5, ax=ax,
)
ax.set_title("Длина текста по классам (без выбросов)")
ax.set_xlabel("Класс тональности")
ax.set_ylabel("Длина текста (символы)")
plt.tight_layout()
plt.show()

---
## 4. Частотный анализ слов

Удаляем русские стоп-слова, знаки препинания и приводим к нижнему регистру.

In [ ]:
# Расширенный список русских стоп-слов
RUSSIAN_STOPWORDS = {
    "и", "в", "во", "не", "что", "он", "на", "я", "с", "со", "как", "а", "то",
    "все", "она", "так", "его", "но", "да", "ты", "к", "у", "же", "вы", "за",
    "бы", "по", "только", "её", "ее", "мне", "было", "вот", "от", "меня", "ещё",
    "еще", "нет", "о", "из", "ему", "теперь", "когда", "даже", "ну", "вдруг",
    "ли", "если", "уже", "или", "ни", "быть", "был", "него", "до", "вас", "нибудь",
    "опять", "уж", "вам", "ведь", "там", "потом", "себя", "ей", "ничего", "перед",
    "более", "тоже", "мой", "будет", "чем", "их", "для", "при", "моя", "мы",
    "этот", "этого", "этом", "это", "этой", "эти", "этих", "эту",
    "чтобы", "без", "будто", "чуть", "раз", "тут", "им", "них",
    "где", "есть", "надо", "ней", "для", "мы", "тебя", "нас", "чего",
    "раз", "нём", "нем", "какой", "были", "один", "тогда", "можно",
    "после", "над", "больше", "том", "через", "очень", "себе", "ней",
    "про", "под", "об", "такой", "мой", "моё", "мое", "моя", "мои",
    "наш", "наша", "наше", "наши", "ваш", "ваша", "ваше", "ваши",
    "свой", "своя", "своё", "свое", "свои", "свою",
    "кто", "тот", "та", "те", "тех", "тем", "той", "того",
    "всё", "всего", "всем", "всех", "всю", "вся",
    "какая", "какие", "какого", "какой", "каком", "каких",
    "сам", "сама", "само", "сами", "которые", "которая", "который", "которого",
    "чтоб", "между", "будут", "была", "может", "нельзя",
    "хоть", "куда", "здесь", "пока", "лишь",
    "другой", "другая", "другие", "другого",
}


def tokenize(text: str) -> list[str]:
    """Приведение к нижнему регистру, удаление всего кроме кириллицы, разбивка на слова."""
    text = text.lower()
    text = re.sub(r"[^а-яёА-ЯЁ\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if t not in RUSSIAN_STOPWORDS and len(t) > 2]


df["tokens"] = df["text"].apply(tokenize)
print(f"Пример токенизации:")
print(f"  Текст: {df['text'].iloc[0][:100]}...")
print(f"  Токены: {df['tokens'].iloc[0]}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, label in zip(axes, SENTIMENT_ORDER):
    all_tokens = [tok for tokens in df[df["sentiment"] == label]["tokens"] for tok in tokens]
    top20 = Counter(all_tokens).most_common(20)
    words, counts = zip(*top20)

    ax.barh(
        range(len(words)), counts,
        color=SENTIMENT_COLORS[label], edgecolor="white", linewidth=0.5,
    )
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=10)
    ax.invert_yaxis()
    ax.set_title(f"Топ-20 слов: {label}", fontweight="bold")
    ax.set_xlabel("Частота")

plt.suptitle("Самые частые слова по классам (без стоп-слов)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Анализ биграмм

Топ биграммы (пары слов) для каждого класса тональности.

In [ ]:
def get_bigrams(tokens: list[str]) -> list[tuple[str, str]]:
    return list(zip(tokens, tokens[1:]))


df["bigrams"] = df["tokens"].apply(get_bigrams)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, label in zip(axes, SENTIMENT_ORDER):
    all_bigrams = [bg for bgs in df[df["sentiment"] == label]["bigrams"] for bg in bgs]
    top15 = Counter(all_bigrams).most_common(15)
    bigram_labels = [f"{a} {b}" for (a, b), _ in top15]
    counts = [c for _, c in top15]

    ax.barh(
        range(len(bigram_labels)), counts,
        color=SENTIMENT_COLORS[label], edgecolor="white", linewidth=0.5,
    )
    ax.set_yticks(range(len(bigram_labels)))
    ax.set_yticklabels(bigram_labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(f"Топ-15 биграмм: {label}", fontweight="bold")
    ax.set_xlabel("Частота")

plt.suptitle("Самые частые биграммы по классам", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Количество предложений в тексте

In [ ]:
def count_sentences(text: str) -> int:
    """Простая эвристика: считаем разделители предложений (.!?…)"""
    parts = re.split(r"[.!?…]+", text.strip())
    # убираем пустые строки от trailing разделителей
    return max(1, len([p for p in parts if p.strip()]))


df["sentence_count"] = df["text"].apply(count_sentences)

print("Статистика количества предложений:")
print(df["sentence_count"].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- гистограмма ---
for label in SENTIMENT_ORDER:
    subset = df[df["sentiment"] == label]["sentence_count"]
    axes[0].hist(
        subset, bins=range(1, df["sentence_count"].max() + 2),
        alpha=0.55, label=label, color=SENTIMENT_COLORS[label],
        edgecolor="white", linewidth=0.5,
    )
axes[0].set_title("Распределение количества предложений")
axes[0].set_xlabel("Количество предложений")
axes[0].set_ylabel("Количество текстов")
axes[0].legend()

# --- boxplot ---
sns.boxplot(
    data=df, x="sentiment", y="sentence_count", order=SENTIMENT_ORDER,
    palette=SENTIMENT_COLORS, showfliers=False, width=0.5, ax=axes[1],
)
axes[1].set_title("Количество предложений по классам")
axes[1].set_xlabel("Класс тональности")
axes[1].set_ylabel("Количество предложений")

plt.tight_layout()
plt.show()

---
## 7. Анализ дисбаланса классов

In [ ]:
majority_class = class_counts.idxmax()
minority_class = class_counts.idxmin()
imbalance_ratio = class_counts.max() / class_counts.min()

print(f"Мажоритарный класс: '{majority_class}' ({class_counts.max():,} записей)")
print(f"Миноритарный класс: '{minority_class}' ({class_counts.min():,} записей)")
print(f"Коэффициент дисбаланса (max/min): {imbalance_ratio:.2f}x")
print()

if imbalance_ratio < 1.5:
    severity = "НИЗКИЙ"
    color = "\033[92m"  # green
elif imbalance_ratio < 3.0:
    severity = "УМЕРЕННЫЙ"
    color = "\033[93m"  # yellow
else:
    severity = "ВЫСОКИЙ"
    color = "\033[91m"  # red

print(f"Уровень дисбаланса: {color}{severity}\033[0m")
print()
print("Рекомендации:")
if imbalance_ratio < 2.0:
    print("  - Дисбаланс незначительный. Можно обучать модель без дополнительных мер.")
    print("  - При желании можно использовать class_weight='balanced' для тонкой настройки.")
    print("  - Метрика: macro F1-score (учитывает все классы равнозначно).")
else:
    print("  - Использовать class_weight='balanced' или focal loss.")
    print("  - Рассмотреть oversampling миноритарного класса.")
    print("  - Обязательно использовать stratified split для train/val/test.")
    print("  - Метрика: macro F1-score.")

In [ ]:
# Визуализация: сравнение текущего распределения с идеально сбалансированным
ideal_count = len(df) // 3

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(SENTIMENT_ORDER))
width = 0.35

bars_actual = ax.bar(
    x - width / 2, [class_counts[s] for s in SENTIMENT_ORDER],
    width, label="Реальное",
    color=[SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER],
    edgecolor="white", linewidth=1.2,
)
bars_ideal = ax.bar(
    x + width / 2, [ideal_count] * 3,
    width, label="Идеальное (равномерное)",
    color="#bdc3c7", edgecolor="white", linewidth=1.2, alpha=0.7,
)

ax.set_xticks(x)
ax.set_xticklabels(SENTIMENT_ORDER)
ax.set_title("Реальное vs идеальное распределение классов")
ax.set_ylabel("Количество записей")
ax.legend()

for bar in bars_actual:
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
        f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=9, fontweight="bold",
    )

plt.tight_layout()
plt.show()

---
## 8. Качество данных

In [ ]:
print("=" * 60)
print("ПРОВЕРКА КАЧЕСТВА ДАННЫХ")
print("=" * 60)

# --- Пропуски ---
nulls = df[["text", "sentiment"]].isnull().sum()
print(f"\n1) Пропущенные значения:")
print(f"   text:      {nulls['text']}")
print(f"   sentiment: {nulls['sentiment']}")

# --- Пустые строки ---
empty_texts = (df["text"].str.strip() == "").sum()
print(f"\n2) Пустые тексты (только пробелы): {empty_texts}")

# --- Дубликаты ---
n_dup_text = df["text"].duplicated().sum()
n_dup_full = df.duplicated().sum()
print(f"\n3) Дубликаты:")
print(f"   Полные дубликаты (text + sentiment): {n_dup_full}")
print(f"   Дубликаты только по тексту:          {n_dup_text}")

# --- Противоречивые метки ---
if n_dup_text > n_dup_full:
    conflicting = n_dup_text - n_dup_full
    print(f"   Тексты с разными метками:           {conflicting}")
    # покажем примеры
    dup_texts = df[df["text"].duplicated(keep=False)].sort_values("text")
    conflicts = dup_texts.groupby("text")["sentiment"].nunique()
    conflict_texts = conflicts[conflicts > 1].index[:3]
    if len(conflict_texts) > 0:
        print("\n   Примеры противоречий:")
        for txt in conflict_texts:
            labels = dup_texts[dup_texts["text"] == txt]["sentiment"].unique()
            print(f"     '{txt[:80]}...' -> {list(labels)}")

# --- Очень короткие тексты ---
very_short = df[df["text_len"] < 10]
print(f"\n4) Очень короткие тексты (< 10 символов): {len(very_short)}")
if len(very_short) > 0:
    print("   Примеры:")
    for _, row in very_short.head(5).iterrows():
        print(f"     [{row['sentiment']}] '{row['text']}'")

# --- Очень длинные тексты ---
p99 = df["text_len"].quantile(0.99)
very_long = df[df["text_len"] > p99]
print(f"\n5) Очень длинные тексты (> {p99:.0f} символов, 99-й перцентиль): {len(very_long)}")
if len(very_long) > 0:
    print(f"   Максимальная длина: {df['text_len'].max()} символов")

# --- Невалидные метки ---
valid_labels = {"позитив", "негатив", "нейтрально"}
invalid = df[~df["sentiment"].isin(valid_labels)]
print(f"\n6) Невалидные метки тональности: {len(invalid)}")
if len(invalid) > 0:
    print(f"   Найденные значения: {invalid['sentiment'].unique().tolist()}")

print(f"\n{'=' * 60}")

In [ ]:
# Визуализация: распределение длин с выделением зон коротких/длинных текстов
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(df["text_len"], bins=80, color="#3498db", edgecolor="white", linewidth=0.5, alpha=0.8)

# отметим зоны
ax.axvline(10, color="#e74c3c", linestyle="--", linewidth=1.5, label="Порог: < 10 символов")
ax.axvline(p99, color="#e67e22", linestyle="--", linewidth=1.5, label=f"99-й перцентиль ({p99:.0f} симв.)")

ax.set_title("Общее распределение длины текста с зонами качества")
ax.set_xlabel("Длина текста (символы)")
ax.set_ylabel("Количество")
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Сводная таблица статистик

In [ ]:
summary = (
    df.groupby("sentiment")
    .agg(
        count=("text", "size"),
        mean_char_len=("text_len", "mean"),
        median_char_len=("text_len", "median"),
        std_char_len=("text_len", "std"),
        min_char_len=("text_len", "min"),
        max_char_len=("text_len", "max"),
        mean_word_count=("word_count", "mean"),
        median_word_count=("word_count", "median"),
        std_word_count=("word_count", "std"),
        mean_sent_count=("sentence_count", "mean"),
        median_sent_count=("sentence_count", "median"),
    )
    .reindex(SENTIMENT_ORDER)
    .round(1)
)

summary.columns = [
    "Кол-во",
    "Ср. длина (симв.)", "Медиана (симв.)", "Ст. откл. (симв.)",
    "Мин. (симв.)", "Макс. (симв.)",
    "Ср. слов", "Медиана слов", "Ст. откл. слов",
    "Ср. предложений", "Медиана предл.",
]

summary.index.name = "Класс"
summary

In [ ]:
# Тепловая карта статистик (нормализованная для наглядности)
summary_numeric = summary.drop(columns=["Кол-во"]).astype(float)
summary_norm = (summary_numeric - summary_numeric.min()) / (summary_numeric.max() - summary_numeric.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 3.5))
sns.heatmap(
    summary_norm, annot=summary_numeric.values, fmt=".1f",
    cmap="YlOrRd", linewidths=1, linecolor="white",
    xticklabels=summary_numeric.columns, yticklabels=SENTIMENT_ORDER,
    ax=ax, cbar_kws={"label": "Нормализованное значение"},
)
ax.set_title("Сводная статистика по классам (тепловая карта)", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 10. Выводы

### Основные результаты анализа

**Размер и структура датасета:**
- Датасет содержит ~10 000 записей с двумя полями: текст отзыва и метка тональности (3 класса).
- Данные представляют русскоязычные отзывы о брендах и сервисах (банки, операторы связи, маркетплейсы и др.).

**Распределение классов:**
- Класс «негатив» является мажоритарным (~42%), «нейтрально» и «позитив» представлены примерно одинаково (~29% каждый).
- Дисбаланс умеренный (коэффициент ~1.5x) — некритичный, но рекомендуется использовать `class_weight='balanced'` и stratified split.

**Длина текстов:**
- Тексты достаточно компактные — типичный отзыв содержит 1-4 предложения.
- Распределение длин схоже между классами, что упрощает моделирование.

**Лексические особенности:**
- Негативные тексты чаще содержат слова об уходе к конкурентам, жалобы на сервис.
- Позитивные тексты характеризуются упоминаниями удобства, качества, благодарности.
- Нейтральные тексты чаще описывают факты и информацию без явной оценки.

**Качество данных:**
- Необходимо проверить и при необходимости удалить дубликаты.
- Противоречивые метки (один текст — разные классы) требуют ручной проверки.

### Рекомендации для моделирования

1. **Предобработка:** Минимальная — модели на основе трансформеров (ruBERT, ruRoBERTa) работают лучше с оригинальным текстом.
2. **Разбиение:** Stratified train/val/test split (80/10/10).
3. **Метрика:** Macro F1-score как основная, per-class F1 для детального анализа.
4. **Дисбаланс:** Использовать `class_weight='balanced'` в функции потерь.
5. **Базовая модель:** Fine-tune ruRoBERTa-large на данной задаче.

In [ ]:
print("EDA завершён.")
print(f"Всего записей: {len(df):,}")
print(f"Классов: {df['sentiment'].nunique()}")
print(f"Коэффициент дисбаланса: {imbalance_ratio:.2f}x")
print(f"Средняя длина текста: {df['text_len'].mean():.0f} символов / {df['word_count'].mean():.0f} слов")